# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name: Aixuan Liu
Date: 2026-08-18

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: e:\研究生\bootcamp\bootcamp_aixuan_liu\homework\homework04

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? False


## Helpers (use or modify)

In [4]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [6]:
SYMBOL = 'AAPL'

USE_ALPHA = bool(
    os.getenv('ALPHAVANTAGE_API_KEY')
)

if USE_ALPHA:

    url = 'https://www.alphavantage.co/query'

    params = {
        'function': 'TIME_SERIES_DAILY_ADJUSTED',
        'symbol': SYMBOL,
        'outputsize': 'compact',
        'apikey': os.getenv(
            'ALPHAVANTAGE_API_KEY'
        )
    }

    r = requests.get(
        url,
        params=params,
        timeout=30
    )

    r.raise_for_status()

    js = r.json()

    key = [
        k for k in js
        if 'Time Series' in k
    ][0]

    df_api = (
        pd.DataFrame(js[key])
        .T
        .reset_index()
        .rename(
            columns={
                'index': 'date',
                '5. adjusted close': 'adj_close'
            }
        )[
            ['date', 'adj_close']
        ]
    )

    df_api['date'] = pd.to_datetime(
        df_api['date'],
        errors='coerce'
    )

    df_api['adj_close'] = pd.to_numeric(
        df_api['adj_close'],
        errors='coerce'
    )

else:

    import yfinance as yf

    df_api = yf.download(
        SYMBOL,
        period='3mo',
        interval='1d',
        auto_adjust=False,
        progress=False
    ).reset_index()

    # yfinance may return MultiIndex columns
    if isinstance(df_api.columns, pd.MultiIndex):
        df_api.columns = [
            col[0] if isinstance(col, tuple) else col
            for col in df_api.columns
        ]

    df_api = df_api[
        ['Date', 'Adj Close']
    ].copy()

    df_api.columns = [
        'date',
        'adj_close'
    ]

    df_api['date'] = pd.to_datetime(
        df_api['date'],
        errors='coerce'
    )

    df_api['adj_close'] = pd.to_numeric(
        df_api['adj_close'],
        errors='coerce'
    )


v_api = validate(
    df_api,
    ['date', 'adj_close']
)

v_api

{'missing': [], 'shape': (63, 2), 'na_total': 0}

In [7]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data\raw\api_source-yfinance_symbol-AAPL_20260818-181234.csv


In [10]:
display(df_api.head())

print("\nShape:")
print(df_api.shape)

print("\nData types:")
print(df_api.dtypes)

print("\nMissing values:")
print(df_api.isna().sum())

,date,adj_close
0,2026-05-19,298.712372
1,2026-05-20,301.989563
2,2026-05-21,304.727173
3,2026-05-22,308.553894
4,2026-05-26,308.064301



Shape:
(63, 2)

Data types:
date         datetime64[ns]
adj_close           float64
dtype: object

Missing values:
date         0
adj_close    0
dtype: int64


In [11]:
required_api_columns = [
    'date',
    'adj_close'
]

assert not df_api.empty, \
    "API dataframe is empty."

assert set(
    required_api_columns
).issubset(df_api.columns), \
    "Required API columns are missing."

assert df_api['date'].notna().all(), \
    "Some dates could not be parsed."

assert df_api['adj_close'].notna().all(), \
    "Some adjusted close values could not be parsed."

assert (
    df_api['adj_close'] > 0
).all(), \
    "Adjusted close prices should be positive."

print("API validation passed.")

API validation passed.


In [12]:
if USE_ALPHA:
    print(
        "API source: Alpha Vantage"
    )
else:
    print(
        "API source: Yahoo Finance via yfinance"
    )

print(
    "Ticker:",
    SYMBOL
)

API source: Yahoo Finance via yfinance
Ticker: AAPL


In [13]:
def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join(
        [
            f"{k}-{v}"
            for k, v in meta.items()
        ]
    )

    path = RAW / (
        f"{prefix}_{mid}_{ts()}.csv"
    )

    df.to_csv(
        path,
        index=False
    )

    print(
        'Saved',
        path
    )

    return path

In [14]:
API_SOURCE = (
    'alphavantage'
    if USE_ALPHA
    else 'yfinance'
)

api_path = save_csv(
    df_api.sort_values('date'),
    prefix='api',
    source=API_SOURCE,
    symbol=SYMBOL
)

api_path

Saved data\raw\api_source-yfinance_symbol-AAPL_20260818-234147.csv


WindowsPath('data/raw/api_source-yfinance_symbol-AAPL_20260818-234147.csv')

In [15]:
print(
    "Saved file:",
    api_path
)

print(
    "File exists:",
    api_path.exists()
)

Saved file: data\raw\api_source-yfinance_symbol-AAPL_20260818-234147.csv
File exists: True


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [8]:
SCRAPE_URL = 'https://example.com/markets-table'  # TODO: replace with permitted page
headers = {'User-Agent':'AFE-Homework/1.0'}
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30); resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print('Scrape failed, using inline demo table:', e)
    html = '<table><tr><th>Ticker</th><th>Price</th></tr><tr><td>AAA</td><td>101.2</td></tr></table>'
    soup = BeautifulSoup(html, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)

if 'Price' in df_scrape.columns:
    df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')
v_scrape = validate(df_scrape, list(df_scrape.columns)); v_scrape

Scrape failed, using inline demo table: 404 Client Error: Not Found for url: https://example.com/markets-table


{'missing': [], 'shape': (1, 2), 'na_total': 0}

In [9]:
_ = save_csv(df_scrape, prefix='scrape', site='example', table='markets')

Saved data\raw\scrape_site-example_table-markets_20260818-181447.csv


In [16]:
SCRAPE_URL = (
    'https://en.wikipedia.org/'
    'wiki/List_of_S%26P_500_companies'
)

SCRAPE_URL

'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

In [17]:
headers = {
    'User-Agent':
        'Mozilla/5.0 AFE-Homework/1.0'
}

resp = requests.get(
    SCRAPE_URL,
    headers=headers,
    timeout=30
)

resp.raise_for_status()

print(
    "Status code:",
    resp.status_code
)

Status code: 200


In [18]:
soup = BeautifulSoup(
    resp.text,
    'html.parser'
)

table = soup.find(
    'table',
    id='constituents'
)

if table is None:
    raise ValueError(
        "Could not find "
        "the S&P 500 constituents table."
    )

print(
    "Table found:",
    table is not None
)

Table found: True


In [19]:
rows = []

for tr in table.find_all('tr'):

    cells = tr.find_all(
        ['th', 'td']
    )

    row = [
        cell.get_text(
            ' ',
            strip=True
        )
        for cell in cells
    ]

    if row:
        rows.append(row)

header = rows[0]
data = rows[1:]

df_scrape = pd.DataFrame(
    data,
    columns=header
)

display(
    df_scrape.head()
)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,0000091142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin , Ireland",2011-07-06,0001467373,1989


In [20]:
print(
    "Columns:"
)

print(
    df_scrape.columns.tolist()
)

print(
    "\nShape:"
)

print(
    df_scrape.shape
)

print(
    "\nMissing values:"
)

print(
    df_scrape.isna().sum()
)

Columns:
['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']

Shape:
(503, 8)

Missing values:
Symbol                   0
Security                 0
GICS Sector              0
GICS Sub-Industry        0
Headquarters Location    0
Date added               0
CIK                      0
Founded                  0
dtype: int64


In [21]:
if 'Date added' in df_scrape.columns:

    df_scrape[
        'Date added'
    ] = pd.to_datetime(
        df_scrape[
            'Date added'
        ],
        errors='coerce'
    )


if 'CIK' in df_scrape.columns:

    df_scrape[
        'CIK'
    ] = pd.to_numeric(
        df_scrape[
            'CIK'
        ],
        errors='coerce'
    )


print(
    df_scrape.dtypes
)

Symbol                           object
Security                         object
GICS Sector                      object
GICS Sub-Industry                object
Headquarters Location            object
Date added               datetime64[ns]
CIK                               int64
Founded                          object
dtype: object


In [22]:
required_scrape_columns = [
    'Symbol',
    'Security',
    'GICS Sector'
]

v_scrape = validate(
    df_scrape,
    required_scrape_columns
)

v_scrape

{'missing': [], 'shape': (503, 8), 'na_total': 0}

In [23]:
assert not df_scrape.empty, \
    "Scraped dataframe is empty."

assert set(
    required_scrape_columns
).issubset(
    df_scrape.columns
), \
    "Required scraped columns are missing."

assert (
    df_scrape['Symbol']
    .astype(str)
    .str.strip()
    .ne('')
    .all()
), \
    "Some ticker symbols are blank."

assert (
    df_scrape['Security']
    .astype(str)
    .str.strip()
    .ne('')
    .all()
), \
    "Some company names are blank."

print(
    "Scrape validation passed."
)

Scrape validation passed.


In [24]:
scrape_path = save_csv(
    df_scrape,
    prefix='scrape',
    site='wikipedia',
    table='sp500_constituents'
)

scrape_path

Saved data\raw\scrape_site-wikipedia_table-sp500_constituents_20260819-001545.csv


WindowsPath('data/raw/scrape_site-wikipedia_table-sp500_constituents_20260819-001545.csv')

In [25]:
print(
    "API CSV exists:",
    api_path.exists()
)

print(
    "Scrape CSV exists:",
    scrape_path.exists()
)

API CSV exists: True
Scrape CSV exists: True


## Documentation

### API Source

- Ticker: AAPL
- Primary API: Alpha Vantage when `ALPHAVANTAGE_API_KEY` is available.
- Fallback source: Yahoo Finance via `yfinance`.
- Alpha Vantage function: `TIME_SERIES_DAILY_ADJUSTED`
- Alpha Vantage output size: `compact`
- Yahoo Finance period: 3 months
- Yahoo Finance interval: 1 day
- Fields retained: `date` and `adj_close`

### API Validation

The API dataset is validated by:

- checking that `date` and `adj_close` exist;
- checking dataframe shape;
- checking missing values;
- converting `date` to datetime;
- converting `adj_close` to numeric;
- confirming the dataframe is not empty;
- confirming adjusted closing prices are positive.

### Scrape Source

- Source: Wikipedia
- Page: List of S&P 500 companies
- URL: https://en.wikipedia.org/wiki/List_of_S%26P_500_companies
- Table: S&P 500 constituents
- Parsing tools: `requests` and `BeautifulSoup`

### Scrape Validation

The scraped dataset is validated by:

- checking required columns: `Symbol`, `Security`, and `GICS Sector`;
- checking dataframe shape;
- checking missing values;
- validating text columns;
- converting `Date added` to datetime when available;
- converting `CIK` to numeric when available.

### Assumptions & Risks

- Internet access is required for both data sources.
- Alpha Vantage may enforce API rate limits.
- If an Alpha Vantage API key is unavailable, the notebook uses Yahoo Finance as a fallback.
- API schemas may change over time.
- The Wikipedia HTML structure may change, which could break the `table#constituents` selector.
- Missing values may exist in some source fields.
- Timestamped filenames are used so raw files are not overwritten.

### Secrets

The `.env` file is stored locally and must not be committed to GitHub.

The `.env.example` file is included in the repository as a template and should not contain a real API key.